In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

### 대본 불러오기 및 포맷팅

In [ ]:
def load_and_format_script(csv_path: str) -> str:
    df = pd.read_csv(csv_path, sep='|', encoding='utf-8')
    
    script_lines = []
    for _, row in df.iterrows():
        if pd.notna(row['speaker_id']):
            speaker_num = str(row['speaker_id']).replace("SPEAKER_", "")
            speaker = f"화자_{speaker_num}"
        else:
            speaker = "알수없음"
            
        text = str(row['text']).strip()
        if text and text != "nan":
            script_lines.append(f"{speaker}: {text}")
            
    return "\n".join(script_lines)

### 대본 문맥 교정 (Correction)

In [ ]:
def correct_script(raw_script: str, model_name: str = "gpt-5.6-luna") -> str:
    # 🌟 수정: temperature 파라미터 완전 제거
    llm = ChatOpenAI(model=model_name)
    
    prompt = PromptTemplate.from_template(
        "다음은 음성 인식(STT)을 통해 추출된 대화 대본입니다. "
        "문맥상 어색한 부분, 오탈자, 잘못 인식된 단어를 자연스럽게 교정해주세요.\n\n"
        "[원본 대본]\n{script}\n\n"
        "[교정된 대본]:"
    )
    
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"script": raw_script})

### 핵심 내용 요약 (Summarization)

In [ ]:
def summarize_script(meeting_note_txt: str, model_name: str = "gpt-5.6-luna") -> str:
    # 🌟 수정: temperature 파라미터 완전 제거
    llm = ChatOpenAI(model=model_name)
    
    prompt = PromptTemplate.from_template(
        "너는 회의 내용을 요약하는 봇이다. 아래 회의록을 읽고, 주요 내용을 요약하라.\n"
        "결과는 마크다운 형식으로 작성한다.\n"
        "아래 형식에 맞추어 작성하라.\n\n"
        "# 회의 제목\n"
        "## 주요 내용\n"
        "## 참석자별 입장\n"
        "## 결정 사항\n\n"
        "=============== 이하 회의록 ===============\n"
        "{meeting_note_txt}"
    )
    
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"meeting_note_txt": meeting_note_txt})

### 전체 파이프라인 실행 (Main)

In [ ]:
def main():
    csv_file_path = "audio/싼기타_비싼기타_final.csv"
    
    print("1. 대본 데이터를 불러오고 포맷을 맞춥니다...")
    raw_script = load_and_format_script(csv_file_path)
    
    print("\n2. LLM을 사용하여 대본의 오탈자를 교정 중입니다...")
    corrected_script = correct_script(raw_script)
    
    with open("audio/싼기타_비싼기타_corrected.txt", "w", encoding="utf-8") as f:
        f.write(corrected_script)
    print(" => 교정된 대본 저장 완료 (audio/싼기타_비싼기타_corrected.txt)")
    
    print("\n3. 교정된 대본을 바탕으로 핵심 내용을 요약 중입니다...")
    summary = summarize_script(corrected_script)
    
    with open("audio/싼기타_비싼기타_summary.md", "w", encoding="utf-8") as f:
        f.write(summary)
    print(" => 회의록 요약본 저장 완료 (audio/싼기타_비싼기타_summary.md)")
    
    print("\n" + "="*50)
    print("[최종 요약 결과]")
    print("="*50)
    print(summary)

### 코드 실행

In [6]:
if __name__ == "__main__":
    main()

1. 대본 데이터를 불러오고 포맷을 맞춥니다...

2. LLM을 사용하여 대본의 오탈자를 교정 중입니다...

3. 교정된 대본을 바탕으로 핵심 내용을 요약 중입니다...

[최종 요약 결과]
# 기타 선택에 대한 토론
## 주요 내용
- 역할극 형식으로 싼 기타와 비싼 기타 중 어느 것으로 시작하는 것이 좋은지에 대한 토론 진행.
- 화자_01은 싼 기타로 시작하는 것이 좋다는 입장을 주장하며, 초보자가 부담 없이 연습할 수 있다고 강조.
- 화자_00은 비싼 기타를 먼저 사는 것이 이중 지출을 막고, 연습의 동기 부여가 된다고 반박.
- 두 화자는 서로의 입장을 존중하며, 각자의 의견을 유머러스하게 교환함.

## 참석자별 입장
- **화자_00 (비싼 기타 주장)**: 비싼 기타를 사면 이중 지출을 피할 수 있으며, 더 좋은 연주 경험과 동기 부여가 된다고 주장.
- **화자_01 (싼 기타 주장)**: 초보자는 부담 없이 연습할 수 있는 싼 기타로 시작하는 것이 좋으며, 나중에 흥미가 생기면 비싼 기타로 업그레이드하는 것이 이상적이라고 주장.

## 결정 사항
- 기타 선택은 개인의 스타일과 상황에 따라 다르며, 가장 중요한 것은 꾸준한 연습과 열정이라는 공감대 형성.
- 비싸고 싼 기타 각각의 장단점이 있으며, 결국 개인의 선택에 따라 다를 수 있음을 인정.

요약 결과가 'audio/싼기타_비싼기타_summary.txt'에 저장되었습니다!
